Day 6 — Loss Functions, Backpropagation & Optimisers
Brain Tumour Detection Project
=====================================
Topics covered:
  1.  CrossEntropyLoss — manual derivation and verification
  2.  Class weights — handling imbalanced MRI dataset
  3.  Backpropagation — chain rule through the full network
  4.  Gradient computation — what .backward() actually does
  5.  SGD vs Adam — why Adam is better for medical imaging
  6.  Adam internals — moment estimates, adaptive learning rate
  7.  Weight decay — L2 regularisation built into the optimiser
  8.  Learning rate scheduler — CosineAnnealingLR
  9.  Gradient clipping — preventing exploding gradients
  

In [76]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
os.makedirs("outputs", exist_ok=True)
torch.manual_seed(42)

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        padding   = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size,
                              stride=stride, padding=padding, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        nn.init.kaiming_normal_(self.conv.weight,
                                mode='fan_in', nonlinearity='relu')
        nn.init.ones_(self.bn.weight)
        nn.init.zeros_(self.bn.bias)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))
 
class BrainTumourCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = ConvBlock(1,   32)
        self.block2 = ConvBlock(32,  64)
        self.block3 = ConvBlock(64,  128)
        self.block4 = ConvBlock(128, 256)
        self.pool   = nn.MaxPool2d(2, 2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
    def forward(self, x):
        x = self.pool(self.block1(x))
        x = self.pool(self.block2(x))
        x = self.pool(self.block3(x))
        x = self.block4(x)
        x = self.gap(x)
        return x.view(x.size(0), -1)
 
class MLPHead(nn.Module):
    def __init__(self, in_features=256, num_classes=4,
                 dropout1=0.4, dropout2=0.3):
        super().__init__()
        self.block1     = nn.Sequential(
            nn.Linear(in_features, 128), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout1))
        self.block2     = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(p=dropout2))
        self.classifier = nn.Linear(64, num_classes)
    def forward(self, x):
        return self.classifier(self.block2(self.block1(x)))
 
class BrainTumourNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.cnn = BrainTumourCNN()
        self.mlp = MLPHead(256, num_classes)
    def forward(self, x):
        return self.mlp(self.cnn(x))

In [77]:
# 1. CROSSENTROPYLOSS — MANUAL DERIVATION
"""
CrossEntropyLoss for multi-class classification:
 
Step 1 — Softmax: convert logits to probabilities
  p_i = exp(z_i) / Σⱼ exp(z_j)
 
Step 2 — Log: take log of the true class probability
  log(p_true)
 
Step 3 — Negate: loss should decrease, so negate
  loss = -log(p_true)
 
Combined: loss = -log( exp(z_true) / Σⱼ exp(z_j) )
        = -z_true + log(Σⱼ exp(z_j))   ← numerically stable form
 
Intuition:
  If model is confident and correct:  p_true ≈ 1.0 → -log(1.0) = 0       (>0.8)
  If model is uncertain:              p_true ≈ 0.25 → -log(0.25) = 1.386 (>0.4)
  If model is confident and wrong:    p_true ≈ 0.0 → -log(0.0) = ∞       (<0.1)
     otherwise Learning
"""

def manual_cross_entropy(logits: torch.Tensor,
                          target: torch.Tensor) -> torch.Tensor:
    """
    logits: (N, C) raw scores
    target: (N,)  integer class indices
    """
    # Numerically stable log-softmax
    # subtract max before exp to prevent overflow
    shifted     = logits - logits.max(dim=1, keepdim=True).values
    log_softmax = shifted - torch.log(torch.exp(shifted).sum(dim=1, keepdim=True))
 
    # Pick log probability of true class for each sample
    N           = logits.shape[0]
    true_log_p  = log_softmax[torch.arange(N), target]
 
    # Average negative log likelihood over the batch
    return -true_log_p.mean()

torch.manual_seed(0)
logits  = torch.randn(8, 4)
targets = torch.randint(0, 4, (8,))
 
loss_manual  = manual_cross_entropy(logits, targets)
loss_pytorch = nn.CrossEntropyLoss()(logits, targets)
 
print(f"Manual CrossEntropyLoss:  {loss_manual.item():.6f}")
print(f"PyTorch CrossEntropyLoss: {loss_pytorch.item():.6f}")
print(f"Match: {torch.allclose(loss_manual, loss_pytorch, atol=1e-5)}")

Manual CrossEntropyLoss:  0.970354
PyTorch CrossEntropyLoss: 0.970354
Match: True


In [78]:
# 2. CLASS WEIGHTS
"""
If glioma has 3x more samples than pituitary:
  Without weights: model learns to predict glioma more often
                   gets high accuracy by ignoring rare classes
  With weights:    rare class errors penalised more heavily
                   model forced to learn all classes equally
 
Weight formula: w_c = total_samples / (num_classes x count_c)
  Rare class   → large weight  → errors penalised more
  Common class → small weight  → errors penalised less
"""

# Realistic class counts (similar to Kaggle dataset)
class_counts = {"glioma": 826, "meningioma": 822,
                "notumour": 395, "pituitary": 827}
classes      = list(class_counts.keys())
counts       = list(class_counts.values())
total        = sum(counts)
num_classes  = len(classes)
 
weights = torch.tensor([
    total / (num_classes * c) for c in counts
], dtype=torch.float32)
 
print(f"{'Class':<15} {'Count':>8} {'Weight':>10}  {'Penalty':>10}")
print("-" * 48)
for cls, cnt, w in zip(classes, counts, weights):
    bar = "█" * int(w * 10)
    print(f"  {cls:<13} {cnt:>8} {w:>10.4f}  {bar}")
 
print(f"\nTotal samples: {total}")
print(f"\nUsage:")
print(f"  criterion = nn.CrossEntropyLoss(weight=class_weights)")
print(f"  → notumour errors penalised {weights[2]/weights[0]:.1f}× more than glioma errors")
 
# Demonstrate effect of class weights on loss
logits_demo   = torch.tensor([[2.0, 0.5, 0.3, 0.1]])   # predicts glioma
target_glioma = torch.tensor([0])   # true = glioma   (common class)
target_notum  = torch.tensor([2])   # true = notumour  (rare class)
 
loss_no_weight_g  = nn.CrossEntropyLoss()(logits_demo, target_glioma)
loss_no_weight_n  = nn.CrossEntropyLoss()(logits_demo, target_notum)
loss_weighted_g   = nn.CrossEntropyLoss(weight=weights)(logits_demo, target_glioma)
loss_weighted_n   = nn.CrossEntropyLoss(weight=weights)(logits_demo, target_notum)
 
print(f"\nSame logits, wrong on rare class vs common class:")
print(f"                    No weight    With weight")
print(f"  Wrong on glioma:  {loss_no_weight_g.item():.4f}       {loss_weighted_g.item():.4f}")
print(f"  Wrong on notum:   {loss_no_weight_n.item():.4f}       {loss_weighted_n.item():.4f}")
print(f"  → weighted loss penalises notumour error "
      f"{loss_weighted_n.item()/loss_weighted_g.item():.1f}x more")

Class              Count     Weight     Penalty
------------------------------------------------
  glioma             826     0.8686  ████████
  meningioma         822     0.8729  ████████
  notumour           395     1.8165  ██████████████████
  pituitary          827     0.8676  ████████

Total samples: 2870

Usage:
  criterion = nn.CrossEntropyLoss(weight=class_weights)
  → notumour errors penalised 2.1× more than glioma errors

Same logits, wrong on rare class vs common class:
                    No weight    With weight
  Wrong on glioma:  0.4417       0.4417
  Wrong on notum:   2.1417       2.1417
  → weighted loss penalises notumour error 4.8x more


In [79]:
# 3. BACKPROPAGATION — CHAIN RULE
"""
Forward pass:  input → layers → loss
Backward pass: loss → ∂loss/∂weights for every weight
 
Chain rule: if loss = f(g(h(x))), then
  ∂loss/∂x = ∂f/∂g x ∂g/∂h x ∂h/∂x
 
In our network:
  loss = CE( MLP( CNN( image ) ) )
 
  ∂loss/∂W_mlp2   = ∂CE/∂logits x ∂logits/∂W_mlp2
  ∂loss/∂W_mlp1   = ∂CE/∂logits x ∂logits/∂h1 x ∂h1/∂W_mlp1
  ∂loss/∂W_conv4  = ... x ∂h1/∂features x ∂features/∂W_conv4
  ∂loss/∂W_conv1  = ... (chain continues all the way back)
 
PyTorch builds a computation graph during the forward pass.
.backward() traverses this graph in reverse, computing gradients
at each node using the chain rule automatically.
"""

# Show computation graph exists after forward pass
model = BrainTumourNet()
model.train()
x     = torch.randn(2, 1, 128, 128)
labels = torch.tensor([0, 2])
 
logits = model(x)
loss   = nn.CrossEntropyLoss()(logits, labels)
 
print(f"After forward pass:")
print(f"  loss.grad_fn: {loss.grad_fn}")
print(f"  logits.grad_fn: {logits.grad_fn}")
print(f"  (grad_fn = the backward operation PyTorch will call)")


print(f"\nBefore .backward():")
print(f"  model.cnn.block1.conv.weight.grad: "
      f"{model.cnn.block1.conv.weight.grad}")


loss.backward()
print(f"\nAfter .backward():")
g = model.cnn.block1.conv.weight.grad
print(f"  model.cnn.block1.conv.weight.grad shape: {tuple(g.shape)}")
print(f"  gradient mean: {g.abs().mean():.6f}")
print(f"  gradient std:  {g.std():.6f}")

model.zero_grad()   # clear gradients before next use

After forward pass:
  loss.grad_fn: <NllLossBackward0 object at 0x000002358386BEB0>
  logits.grad_fn: <AddmmBackward0 object at 0x000002358386BB80>
  (grad_fn = the backward operation PyTorch will call)

Before .backward():
  model.cnn.block1.conv.weight.grad: None

After .backward():
  model.cnn.block1.conv.weight.grad shape: (32, 1, 3, 3)
  gradient mean: 0.000889
  gradient std:  0.001188


In [80]:
# 4. GRADIENT
"""
grad[i,j] = ∂loss/∂weight[i,j]
 
This tells you:
  "If I increase weight[i,j] by a tiny amount ε,
   the loss will change by grad[i,j] x ε"
 
Positive gradient → increasing weight increases loss
                  → we should DECREASE the weight
 
Negative gradient → increasing weight decreases loss
                  → we should INCREASE the weight
 
Update rule (gradient descent):
  weight = weight - lr x gradient
 
  lr = learning rate — how big a step to take
  Small lr → slow but stable
  Large lr → fast but can overshoot and diverge
"""

torch.manual_seed(1)
model_manual = BrainTumourNet()
model_manual.train()

lr     = 0.01
x_gd   = torch.randn(2, 1, 128, 128)
lbl_gd = torch.tensor([1, 3])

# Store original weight
w_before = model_manual.mlp.classifier.weight.clone()
 
# Forward + backward
out_gd   = model_manual(x_gd)
loss_gd  = nn.CrossEntropyLoss()(out_gd, lbl_gd)
loss_gd.backward()

# Manual update
grad = model_manual.mlp.classifier.weight.grad.clone()
with torch.no_grad():
    model_manual.mlp.classifier.weight -= lr * grad
 
w_after  = model_manual.mlp.classifier.weight.clone()
w_change = (w_after - w_before).abs().mean().item()
 
print(f"Learning rate: {lr}")
print(f"Gradient mean: {grad.abs().mean().item():.6f}")
print(f"Weight change: {w_change:.6f}  (= lr x grad_mean = {lr * grad.abs().mean().item():.6f})")
print(f"Match: {abs(w_change - lr * grad.abs().mean().item()) < 1e-7}")

Learning rate: 0.01
Gradient mean: 0.010846
Weight change: 0.000108  (= lr x grad_mean = 0.000108)
Match: True


In [81]:
# 5. SGD vs ADAM
"""
SGD (Stochastic Gradient Descent):
  weight = weight - lr x gradient
  Same learning rate for every weight, every step
  Problem: some weights need large steps, some need tiny steps
           hard to set one lr that works for all layers
 
Adam (Adaptive Moment estimation):
  Keeps TWO running averages per weight:
    m = 0.9 x m + 0.1 x grad          ← 1st moment (mean of gradients)
    v = 0.999 x v + 0.001 x grad²     ← 2nd moment (mean of grad²)
 
  Bias correction (important early in training when m,v ≈ 0):
    m̂ = m / (1 - 0.9^t)
    v̂ = v / (1 - 0.999^t)
 
  Update:
    weight = weight - lr x m̂ / (√v̂ + ε)
 
  Effect:
    Large gradient history (v large) → small effective lr  (slow down)
    Small gradient history (v small) → large effective lr  (speed up)
    → each weight gets its own adaptive learning rate
    → much faster convergence than SGD on complex tasks
"""

print("Simulating Adam vs SGD on a simple loss landscape:")
print(f"{'Step':>6} {'SGD loss':>12} {'Adam loss':>12}")
print("-" * 34)
 
torch.manual_seed(42)
w_sgd  = torch.tensor([3.0], requires_grad=True)
w_adam = torch.tensor([3.0], requires_grad=True)
 
optim_sgd  = torch.optim.SGD([w_sgd],  lr=0.01)
optim_adam = torch.optim.Adam([w_adam], lr=0.01)
 
# Minimise f(w) = w²+noise (minimum at w=0)
for step in range(1, 11):
    loss_s = (w_sgd) ** 2  
    optim_sgd.zero_grad(); loss_s.backward(); optim_sgd.step()
 
    loss_a = (w_adam) ** 2 
    optim_adam.zero_grad(); loss_a.backward(); optim_adam.step()
 
    if step <= 5 or step == 10:
        print(f"{step:>6} {loss_s.item():>12.6f} {loss_a.item():>12.6f}")
 
print(f"\nSGD converges faster on this simple example. Smooth curve")
print(f"On deep CNNs with many layers the difference is much larger, Adam prevails.")

Simulating Adam vs SGD on a simple loss landscape:
  Step     SGD loss    Adam loss
----------------------------------
     1     9.000000     9.000000
     2     8.643600     8.940100
     3     8.301313     8.880405
     4     7.972581     8.820920
     5     7.656867     8.761647
    10     6.256217     8.468575

SGD converges faster on this simple example. Smooth curve
On deep CNNs with many layers the difference is much larger, Adam prevails.


In [ ]:
# 6. ADAM INTERNALS
"""
Adam, implemented by hand one step at a time and checked against
torch.optim.Adam driven by identical gradients.

  m = B1 x m + (1 - B1) x g          1st moment   (B1 = 0.9)
  v = B2 x v + (1 - B2) x g^2        2nd moment   (B2 = 0.999)

  m_hat = m / (1 - B1^t)             bias correction
  v_hat = v / (1 - B2^t)

  w = w - lr x m_hat / (sqrt(v_hat) + eps)

Why bias correction is needed:
  m and v both start at zero, so early estimates are pulled
  towards zero. At t=1, (1 - 0.9^1) = 0.1, so m_hat = m / 0.1 = 10 x m
  which exactly cancels the (1 - B1) = 0.1 factor that shrank it.
  Without correction Adam would barely move for the first ~10 steps.

If the manual and PyTorch columns match to ~1e-7, the update rule
above is exactly what the optimiser is doing internally.
"""
B1, B2, EPS, LR = 0.9, 0.999, 1e-8, 1e-3

w_manual = torch.tensor([1.0, -0.5, 0.3])
w_torch  = torch.tensor([1.0, -0.5, 0.3], requires_grad=True)

m = torch.zeros_like(w_manual)
v = torch.zeros_like(w_manual)

optim = torch.optim.Adam([w_torch], lr=LR, betas=(B1, B2), eps=EPS)

torch.manual_seed(42)
print(f"Initial weights: {w_manual.tolist()}")
print(f"\n{'Step':>4}  {'manual':>34}  {'max|diff|':>10}")
print("-" * 54)

for t in range(1, 4):
    grad = torch.randn(3)          # same gradient fed to both paths

    # -- manual Adam ------------------------------------------------
    m = B1 * m + (1 - B1) * grad
    v = B2 * v + (1 - B2) * grad ** 2
    m_hat = m / (1 - B1 ** t)      # t is 1-indexed -- the step number
    v_hat = v / (1 - B2 ** t)
    w_manual = w_manual - LR * m_hat / (v_hat.sqrt() + EPS)

    # -- PyTorch Adam -----------------------------------------------
    w_torch.grad = grad.clone()
    optim.step()
    optim.zero_grad()

    diff = (w_manual - w_torch.detach()).abs().max().item()
    vals = ", ".join(f"{x:+.6f}" for x in w_manual.tolist())
    print(f"{t:>4}  [{vals}]  {diff:>10.2e}")

print("\n  -> manual and PyTorch agree to floating-point precision")
print(f"  -> first step moved each weight by ~{LR:.0e} (= lr), because")
print("     m_hat / sqrt(v_hat) ~= 1 when t=1 regardless of gradient size")


In [83]:
# 7. WEIGHT DECAY — L2 REGULARISATION
"""
L2 regularisation adds a penalty to the loss for large weights:
  total_loss = task_loss + λ x Σ weight²
 
This penalises the model for using large weights — which tend
to overfit by memorising training data. Forces the model to
spread information across many small weights rather than
relying heavily on a few large ones.
 
In PyTorch optimisers, weight_decay = λ directly:
  Adam(params, lr=1e-3, weight_decay=1e-4)
 
The update becomes:
  weight = weight - lr x (grad + weight_decay x weight)
                                    ↑
                          pulls weight toward zero each step
                          (that's why it's called "decay")
 
weight_decay=1e-4 is a good default for our model:
  Too small (1e-6): no regularisation effect
  Too large (1e-2): weights collapse to zero, underfitting
"""

lr, wd = 1e-3, 1e-4
w_example  = torch.tensor(0.5)   # a weight with value 0.5
grad_example = torch.tensor(0.1) # its gradient
 
update_no_wd = lr * grad_example
update_wd    = lr * (grad_example + wd * w_example)
decay_contrib = lr * wd * w_example
 
print(f"Weight value:         {w_example.item()}")
print(f"Gradient:             {grad_example.item()}")
print(f"\nUpdate without decay: {update_no_wd.item():.6f}")
print(f"Update with decay:    {update_wd.item():.6f}")
print(f"  (decay contribution: {decay_contrib.item():.6f} — tiny but consistent)")

Weight value:         0.5
Gradient:             0.10000000149011612

Update without decay: 0.000100
Update with decay:    0.000100
  (decay contribution: 0.000000 — tiny but consistent)


In [84]:
# 8. LEARNING RATE SCHEDULER
"""
Fixed learning rate problems:
  Too high: loss oscillates, never converges to minimum
  Too low:  converges but very slowly
 
Solution: start with a higher lr, reduce it over training.
 
CosineAnnealingLR:
  lr(t) = lr_min + 0.5*(lr_max - lr_min)*(1 + cos(π*t/T_max))
 
  Starts at lr_max, smoothly decays following a cosine curve,
  reaches lr_min at T_max epochs.
  No sudden drops — smooth and stable.
 
Why cosine (not linear or step decay)?
  Linear: drops too fast early when model still learning fast
  Step:   sudden drops cause instability
  Cosine: slow decay early, faster in middle, slow near minimum
          naturally matches how learning dynamics work
"""

model_sch  = BrainTumourNet()
optimizer  = torch.optim.Adam(model_sch.parameters(),
                               lr=1e-3, weight_decay=1e-4)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=50, eta_min=1e-6)
 
lrs = []
for epoch in range(50):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()
 
print(f"Learning rate schedule over 50 epochs:")
print(f"  Epoch  1: lr = {lrs[0]:.6f}")
print(f"  Epoch 10: lr = {lrs[9]:.6f}")
print(f"  Epoch 25: lr = {lrs[24]:.6f}")
print(f"  Epoch 40: lr = {lrs[39]:.6f}")
print(f"  Epoch 50: lr = {lrs[49]:.6f}")
 
# Plot LR schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#F8F8F6')
 
axes[0].plot(range(1, 51), lrs, color='#378ADD', lw=2.5)
axes[0].set_title("CosineAnnealingLR schedule\n(50 epochs, lr=1e-3 → 1e-6)",
                   fontsize=10, fontweight='bold')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Learning rate")
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)
 
# Loss curve shape for different lr schedules
epochs  = np.linspace(0, 1, 100)
loss_fixed   = 1.4 * np.exp(-2 * epochs) + 0.15 + 0.05*np.random.randn(100)
loss_cosine  = 1.4 * np.exp(-3 * epochs) + 0.08 + 0.03*np.random.randn(100)
loss_toolarge= np.abs(1.4 - 2*epochs + 0.3*np.random.randn(100))
 
axes[1].plot(loss_fixed,    color='#D85A30', lw=2, label='Fixed lr=1e-3', alpha=0.8)
axes[1].plot(loss_cosine,   color='#1D9E75', lw=2, label='Cosine schedule', alpha=0.8)
axes[1].plot(loss_toolarge, color='gray',    lw=2, label='Fixed lr too large', alpha=0.6)
axes[1].set_title("Training loss shape\nfor different lr strategies",
                   fontsize=10, fontweight='bold')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 2)
axes[1].grid(alpha=0.3)
 
plt.suptitle("Learning rate scheduling", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("outputs/lr_schedule.png", dpi=120, bbox_inches='tight',
            facecolor=fig.get_facecolor())

Learning rate schedule over 50 epochs:
  Epoch  1: lr = 0.001000
  Epoch 10: lr = 0.000922
  Epoch 25: lr = 0.000532
  Epoch 40: lr = 0.000116
  Epoch 50: lr = 0.000002


In [85]:
# 9. GRADIENT CLIPPING
"""
Even with Kaiming init and BatchNorm, gradients can occasionally
spike during training — especially early on or with hard batches.
 
Gradient clipping rescales all gradients if their total norm
exceeds a threshold:
 
  if ||grad|| > max_norm:
      grad = grad x (max_norm / ||grad||)
 
This prevents any single bad batch from causing a huge weight
update that destabilises training.
 
  max_norm = 1.0 is a safe default for our model.
  Applied AFTER loss.backward(), BEFORE optimizer.step().
"""
model_clip = BrainTumourNet()
model_clip.train()
 
x_clip   = torch.randn(4, 1, 128, 128)
lbl_clip = torch.randint(0, 4, (4,))
 
out_clip  = model_clip(x_clip)
loss_clip = nn.CrossEntropyLoss()(out_clip, lbl_clip)
loss_clip.backward()
 
# Compute total gradient norm before clipping
total_norm_before = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        total_norm_before += p.grad.norm(2).item() ** 2
total_norm_before = total_norm_before ** 0.5
 
# Apply gradient clipping
nn.utils.clip_grad_norm_(model_clip.parameters(), max_norm=1.0)
 
# Compute total gradient norm after clipping
total_norm_after = 0.0
for p in model_clip.parameters():
    if p.grad is not None:
        total_norm_after += p.grad.norm(2).item() ** 2
total_norm_after = total_norm_after ** 0.5
 
print(f"Gradient norm before clipping: {total_norm_before:.4f}")
print(f"Gradient norm after clipping:  {total_norm_after:.4f}")
print(f"max_norm = 1.0")
clipped = total_norm_before > 1.0
print(f"Was clipping applied? {clipped}")
if clipped:
    print(f"  Gradients rescaled by factor: {total_norm_after/total_norm_before:.4f}")





Gradient norm before clipping: 1.0924
Gradient norm after clipping:  1.0000
max_norm = 1.0
Was clipping applied? True
  Gradients rescaled by factor: 0.9154


In [86]:
# 10. FULL TRAINING SETUP + VERIFICATION
print("\n" + "=" * 60)
print("10. FULL TRAINING SETUP — EVERYTHING READY FOR DAY 7")
print("=" * 60)
 
model     = BrainTumourNet()
criterion = nn.CrossEntropyLoss(weight=weights)   # class-weighted loss
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4,    # L2 regularisation
    betas=(0.9, 0.999),   # Adam moment decay rates (defaults)
    eps=1e-8,             # numerical stability (default)
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,    # number of training epochs
    eta_min=1e-6 # minimum learning rate
)
 
print("Training components:")
print(f"  Model:      BrainTumourNet ({sum(p.numel() for p in model.parameters()):,} params)")
print(f"  Loss:       CrossEntropyLoss(weight={[round(w.item(),3) for w in weights]})")
print(f"  Optimiser:  Adam(lr=1e-3, weight_decay=1e-4)")
print(f"  Scheduler:  CosineAnnealingLR(T_max=50, eta_min=1e-6)")
print(f"  Clipping:   clip_grad_norm_(max_norm=1.0)")
 
# Simulate one complete training step
print(f"\nSimulating one complete training step:")
model.train()
x_sim   = torch.randn(32, 1, 128, 128)
lbl_sim = torch.randint(0, 4, (32,))
 
optimizer.zero_grad()
logits_sim = model(x_sim)
loss_sim   = criterion(logits_sim, lbl_sim)
loss_sim.backward()
nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
scheduler.step()
 
print(f"  ✓ optimizer.zero_grad()")
print(f"  ✓ logits = model(x)         → {tuple(logits_sim.shape)}")
print(f"  ✓ loss = criterion(logits)  → {loss_sim.item():.4f}")
print(f"  ✓ loss.backward()")
print(f"  ✓ clip_grad_norm_(max=1.0)")
print(f"  ✓ optimizer.step()")
print(f"  ✓ scheduler.step()          → lr = {optimizer.param_groups[0]['lr']:.6f}")


10. FULL TRAINING SETUP — EVERYTHING READY FOR DAY 7
Training components:
  Model:      BrainTumourNet (429,732 params)
  Loss:       CrossEntropyLoss(weight=[0.869, 0.873, 1.816, 0.868])
  Optimiser:  Adam(lr=1e-3, weight_decay=1e-4)
  Scheduler:  CosineAnnealingLR(T_max=50, eta_min=1e-6)
  Clipping:   clip_grad_norm_(max_norm=1.0)

Simulating one complete training step:
  ✓ optimizer.zero_grad()
  ✓ logits = model(x)         → (32, 4)
  ✓ loss = criterion(logits)  → 1.4034
  ✓ loss.backward()
  ✓ clip_grad_norm_(max=1.0)
  ✓ optimizer.step()
  ✓ scheduler.step()          → lr = 0.000999
